# Member C -- Exploration (Epsilon) Experiments

My 10 runs, varying the exploration params (`exploration_initial_eps`,
`exploration_final_eps`, `exploration_fraction`) in different combos.
Everything else stays at the baseline we agreed on in `shared_train.py`
so our results line up with A and B's.

In [ ]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari,accept-rom-license]" ale-py "autorom[accept-rom-license]" pandas
!AutoROM --accept-license

## Pull our repo

This grabs `shared_train.py` and everything else from our repo so the
import below works.

In [ ]:
import os

REPO_DIR = "/content/deep-q-learning-formative3"
if os.path.exists(REPO_DIR):
    # reset --hard (not pull) so this always matches the remote exactly,
    # even if history upstream was rewritten (force-pushed) since your
    # last clone -- a plain pull can't reconcile that.
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone https://github.com/Mikekimm/deep-q-learning-formative3.git {REPO_DIR}
%cd {REPO_DIR}

from shared_train import BASELINE_CONFIG, GAME_ID, TOTAL_TIMESTEPS, SEED, train_one_run
print(GAME_ID, TOTAL_TIMESTEPS, SEED)
print(BASELINE_CONFIG)

## Smoke test -- run this first

Same pipeline as a real run, just 2,000 steps so it only takes a minute
or two. Worth running fresh in every new Colab session -- even though
the code's solid, each session can have its own install quirks. Run it
by itself, confirm it comes back clean, then move on to the real
experiments below (don't Run All the whole notebook).

In [ ]:
import shared_train

_original_total_timesteps = shared_train.TOTAL_TIMESTEPS
shared_train.TOTAL_TIMESTEPS = 2000  # temporary override for this test only

smoke_row = shared_train.train_one_run(overrides={}, run_name="smoketest", member="test")
print(smoke_row)

shared_train.TOTAL_TIMESTEPS = _original_total_timesteps  # restore before real runs below
print("TOTAL_TIMESTEPS restored to", shared_train.TOTAL_TIMESTEPS)

In [ ]:
# 10 combos of (initial_eps, final_eps, fraction of training spent decaying).
# Each changes one dimension of exploration relative to the baseline so the
# effect stays interpretable.
epsilon_combos = [
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.20, "exploration_fraction": 0.05},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.10, "exploration_fraction": 0.05},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.05},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.20},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.30},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.01, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 0.8, "exploration_final_eps": 0.05, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.30, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.50},
]

## Optional: live reward graph

Run this before the loop below if you want to watch reward trends
update live instead of waiting for a run to finish. It's just the
reward graph, not actual gameplay -- we're not rendering the game
during training since it'd slow everything down.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tb_logs

In [ ]:
results = []
for i, combo in enumerate(epsilon_combos, start=1):
    row = train_one_run(
        overrides=combo,
        run_name=f"memberC_eps_{i:02d}",
        member="MemberC",
    )
    results.append(row)